# Telecom Customer Churn Prediction - Comprehensive Analysis
## Course Project: Customer Retention Strategy

**Objective**: Predict customer churn and identify key factors driving churn to develop retention strategies.

## 1. Import Libraries and Load Data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, f1_score
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

ModuleNotFoundError: No module named 'pandas'

In [2]:
# Load the dataset
df = pd.read_csv('/Project_telecom_data.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData Types:")
print(df.dtypes)
print("\nBasic Statistics:")
print(df.describe())

FileNotFoundError: [Errno 2] No such file or directory: '/Project_telecom_data.csv'

## 2. Data Cleaning and Preprocessing

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

# Check for duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Churn distribution
print(f"\nChurn Distribution:")
print(df['churn'].value_counts())
print(f"Churn Rate: {df['churn'].mean()*100:.2f}%")

In [ ]:
# Data cleaning
df_clean = df.copy()

# Fix negative values in numeric columns
print("Checking for negative values in usage columns:")
print(f"Negative calls_made: {(df_clean['calls_made'] < 0).sum()}")
print(f"Negative sms_sent: {(df_clean['sms_sent'] < 0).sum()}")
print(f"Negative data_used: {(df_clean['data_used'] < 0).sum()}")

# Replace negative values with 0 (assuming these are data entry errors)
df_clean[['calls_made', 'sms_sent', 'data_used']] = df_clean[['calls_made', 'sms_sent', 'data_used']].clip(lower=0)
print("\nNegative values replaced with 0")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Churn distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
churn_counts = df_clean['churn'].value_counts()
axes[0].bar(['No Churn', 'Churn'], churn_counts.values, color=['green', 'red'], alpha=0.7)
axes[0].set_title('Churn Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')

# Percentage pie chart
axes[1].pie(churn_counts, labels=['No Churn', 'Churn'], autopct='%1.1f%%', 
            colors=['green', 'red'], startangle=90)
axes[1].set_title('Churn Percentage Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Demographic analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Age vs Churn
df_clean.groupby('age')['churn'].mean().plot(ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Churn Rate by Age', fontweight='bold')
axes[0, 0].set_ylabel('Churn Rate')

# Gender vs Churn
gender_churn = df_clean.groupby('gender')['churn'].agg(['count', 'sum', 'mean'])
axes[0, 1].bar(gender_churn.index, gender_churn['mean'], color=['lightblue', 'lightcoral'])
axes[0, 1].set_title('Churn Rate by Gender', fontweight='bold')
axes[0, 1].set_ylabel('Churn Rate')

# Telecom Partner vs Churn
partner_churn = df_clean.groupby('telecom_partner')['churn'].mean().sort_values(ascending=False)
axes[1, 0].barh(partner_churn.index, partner_churn.values, color='orange')
axes[1, 0].set_title('Churn Rate by Telecom Partner', fontweight='bold')
axes[1, 0].set_xlabel('Churn Rate')

# Dependents vs Churn
dep_churn = df_clean.groupby('num_dependents')['churn'].mean()
axes[1, 1].plot(dep_churn.index, dep_churn.values, marker='o', color='purple', linewidth=2)
axes[1, 1].set_title('Churn Rate by Number of Dependents', fontweight='bold')
axes[1, 1].set_xlabel('Number of Dependents')
axes[1, 1].set_ylabel('Churn Rate')

plt.tight_layout()
plt.show()

In [ ]:
# Usage patterns analysis
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Calls made
churned = df_clean[df_clean['churn'] == 1]['calls_made']
not_churned = df_clean[df_clean['churn'] == 0]['calls_made']
axes[0].hist([not_churned, churned], label=['No Churn', 'Churn'], bins=30, color=['green', 'red'], alpha=0.6)
axes[0].set_title('Calls Made Distribution', fontweight='bold')
axes[0].set_xlabel('Calls Made')
axes[0].legend()

# SMS Sent
churned_sms = df_clean[df_clean['churn'] == 1]['sms_sent']
not_churned_sms = df_clean[df_clean['churn'] == 0]['sms_sent']
axes[1].hist([not_churned_sms, churned_sms], label=['No Churn', 'Churn'], bins=30, color=['green', 'red'], alpha=0.6)
axes[1].set_title('SMS Sent Distribution', fontweight='bold')
axes[1].set_xlabel('SMS Sent')
axes[1].legend()

# Data Used
churned_data = df_clean[df_clean['churn'] == 1]['data_used']
not_churned_data = df_clean[df_clean['churn'] == 0]['data_used']
axes[2].hist([not_churned_data, churned_data], label=['No Churn', 'Churn'], bins=30, color=['green', 'red'], alpha=0.6)
axes[2].set_title('Data Used Distribution', fontweight='bold')
axes[2].set_xlabel('Data Used (MB)')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
correlation_matrix = df_clean[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlation with churn
print("\nCorrelation with Churn:")
print(correlation_matrix['churn'].sort_values(ascending=False))

## 4. Feature Engineering

In [ ]:
# Create a copy for feature engineering
df_features = df_clean.copy()

# Convert date_of_registration to days active
df_features['date_of_registration'] = pd.to_datetime(df_features['date_of_registration'])
reference_date = df_features['date_of_registration'].max()
df_features['days_active'] = (reference_date - df_features['date_of_registration']).dt.days

# Create usage categories
df_features['total_usage'] = df_features['calls_made'] + df_features['sms_sent'] + df_features['data_used']

# Create age groups
df_features['age_group'] = pd.cut(df_features['age'], bins=[0, 30, 40, 50, 60, 100], 
                                   labels=['18-30', '31-40', '41-50', '51-60', '60+'])

# Salary categories
df_features['salary_category'] = pd.qcut(df_features['estimated_salary'], q=3, 
                                          labels=['Low', 'Medium', 'High'], duplicates='drop')

# Usage intensity (high usage if above median)
df_features['high_usage'] = (df_features['total_usage'] > df_features['total_usage'].median()).astype(int)

print("New features created:")
print(df_features[['days_active', 'total_usage', 'age_group', 'salary_category', 'high_usage']].head(10))

## 5. Data Preparation for Modeling

In [ ]:
# Prepare data for modeling
df_model = df_features.copy()

# Drop unnecessary columns
df_model = df_model.drop(['customer_id', 'date_of_registration', 'state', 'city', 'pincode'], axis=1)

# Encode categorical variables
label_encoders = {}
categorical_cols = df_model.select_dtypes(include=['object', 'category']).columns

for col in categorical_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    label_encoders[col] = le

print("Data prepared for modeling:")
print(df_model.head())
print(f"\nShape: {df_model.shape}")
print(f"\nColumns: {df_model.columns.tolist()}")

In [ ]:
# Separate features and target
X = df_model.drop('churn', axis=1)
y = df_model['churn']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train_scaled.shape}")
print(f"Test set size: {X_test_scaled.shape}")
print(f"\nTrain set churn rate: {y_train.mean()*100:.2f}%")
print(f"Test set churn rate: {y_test.mean()*100:.2f}%")

## 6. Model Building and Training

In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

# Train and evaluate models
results = {}

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    print(f"{'='*50}")
    
    # Train the model
    model.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        'f1': f1,
        'auc': auc
    }
    
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print(f"\nF1-Score: {f1:.4f}")
    print(f"AUC-ROC: {auc:.4f}")

## 7. Model Evaluation and Comparison

In [ ]:
# Visualize confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (name, result) in enumerate(results.items()):
    cm = confusion_matrix(y_test, result['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
    axes[idx].set_title(f'{name} - Confusion Matrix', fontweight='bold')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(10, 7))

for name, result in results.items():
    fpr, tpr, _ = roc_curve(y_test, result['y_pred_proba'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={result['auc']:.3f})", linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=2)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Model comparison table
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'F1-Score': [results[name]['f1'] for name in results.keys()],
    'AUC-ROC': [results[name]['auc'] for name in results.keys()]
})

print("\nModel Performance Comparison:")
print(comparison_df.sort_values('AUC-ROC', ascending=False))

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

comparison_df.set_index('Model')[['F1-Score']].plot(kind='bar', ax=axes[0], color='steelblue', legend=False)
axes[0].set_title('F1-Score Comparison', fontweight='bold')
axes[0].set_ylabel('F1-Score')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

comparison_df.set_index('Model')[['AUC-ROC']].plot(kind='bar', ax=axes[1], color='coral', legend=False)
axes[1].set_title('AUC-ROC Comparison', fontweight='bold')
axes[1].set_ylabel('AUC-ROC')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 8. Feature Importance Analysis

In [ ]:
# Get feature importance from Random Forest
rf_model = results['Random Forest']['model']
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

# Visualize feature importance
plt.figure(figsize=(10, 8))
plt.barh(range(len(feature_importance.head(15))), feature_importance.head(15)['Importance'].values, color='teal')
plt.yticks(range(len(feature_importance.head(15))), feature_importance.head(15)['Feature'].values)
plt.xlabel('Importance', fontsize=12)
plt.title('Top 15 Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Logistic Regression coefficients (absolute value for importance)
lr_model = results['Logistic Regression']['model']
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_[0],
    'Abs_Coefficient': np.abs(lr_model.coef_[0])
}).sort_values('Abs_Coefficient', ascending=False)

print("\nTop 10 Logistic Regression Coefficients:")
print(coefficients.head(10))

# Visualize coefficients
plt.figure(figsize=(10, 8))
colors = ['red' if x < 0 else 'green' for x in coefficients.head(15)['Coefficient'].values]
plt.barh(range(len(coefficients.head(15))), coefficients.head(15)['Coefficient'].values, color=colors)
plt.yticks(range(len(coefficients.head(15))), coefficients.head(15)['Feature'].values)
plt.xlabel('Coefficient Value', fontsize=12)
plt.title('Top 15 Features by Coefficient (Logistic Regression)', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Key Insights and Business Recommendations

In [ ]:
# Calculate churn patterns
print("=" * 70)
print("KEY INSIGHTS FROM ANALYSIS")
print("=" * 70)

print("\n1. CHURN STATISTICS:")
print(f"   - Overall Churn Rate: {df_clean['churn'].mean()*100:.2f}%")
print(f"   - Number of Churned Customers: {df_clean['churn'].sum()}")
print(f"   - Number of Retained Customers: {(df_clean['churn']==0).sum()}")

print("\n2. DEMOGRAPHIC INSIGHTS:")
print(f"   - Highest Churn by Gender: {df_clean.groupby('gender')['churn'].mean().idxmax()}")
print(f"   - Highest Churn by Partner: {df_clean.groupby('telecom_partner')['churn'].mean().idxmax()}")
print(f"   - Average Age of Churned: {df_clean[df_clean['churn']==1]['age'].mean():.1f}")
print(f"   - Average Age of Retained: {df_clean[df_clean['churn']==0]['age'].mean():.1f}")

print("\n3. USAGE PATTERNS:")
print(f"   - Avg Calls (Churned): {df_clean[df_clean['churn']==1]['calls_made'].mean():.1f}")
print(f"   - Avg Calls (Retained): {df_clean[df_clean['churn']==0]['calls_made'].mean():.1f}")
print(f"   - Avg Data Usage (Churned): {df_clean[df_clean['churn']==1]['data_used'].mean():.1f}")
print(f"   - Avg Data Usage (Retained): {df_clean[df_clean['churn']==0]['data_used'].mean():.1f}")

print("\n4. MODEL PERFORMANCE:")
for name, result in results.items():
    print(f"   - {name}: AUC = {result['auc']:.4f}, F1 = {result['f1']:.4f}")

print("\n" + "="*70)

In [ ]:
# Business Recommendations
print("\n" + "="*70)
print("BUSINESS RECOMMENDATIONS FOR REDUCING CHURN")
print("="*70)

recommendations = [
    {
        'title': '1. TARGETED RETENTION FOR HIGH-RISK CUSTOMERS',
        'details': [
            '- Focus on customers with low usage patterns (calls, SMS, data)',
            '- Implement early warning system based on usage decline',
            '- Offer usage-based incentives to increase engagement'
        ]
    },
    {
        'title': '2. AGE-BASED STRATEGIES',
        'details': [
            '- Develop age-specific offerings for high-churn age groups',
            '- Senior customers may need dedicated support services',
            '- Younger segments may benefit from value-added services'
        ]
    },
    {
        'title': '3. PARTNER-SPECIFIC INITIATIVES',
        'details': [
            '- Analyze partner performance and align offerings',
            '- Provide incentives for high-churn partners',
            '- Strengthen partnerships with low-churn providers'
        ]
    },
    {
        'title': '4. PROACTIVE CUSTOMER ENGAGEMENT',
        'details': [
            '- Regular personalized offers based on usage patterns',
            '- Customer support initiatives for at-risk segments',
            '- Loyalty programs for long-standing customers'
        ]
    },
    {
        'title': '5. DATA-DRIVEN DECISION MAKING',
        'details': [
            '- Deploy predictive models to identify at-risk customers',
            '- Prioritize high-value customers in retention efforts',
            '- Monitor model performance and update regularly'
        ]
    }
]
,
    for detail in rec['details']:
        print(f"  {detail}")

print("\n" + "="*70)

IndentationError: unexpected indent (3426445110.py, line 49)

## 10. Summary and Conclusion

In [ ]:
summary_text = f"""
PROJECT SUMMARY
{'='*70}

OBJECTIVE:
Develop a predictive model to identify customers at risk of churning and 
provide actionable insights for customer retention strategies.

DATA OVERVIEW:
- Total Customers: {len(df_clean):,}
- Churn Rate: {df_clean['churn'].mean()*100:.2f}%
- Features Used: {X.shape[1]}
- Training Samples: {X_train_scaled.shape[0]:,}
- Test Samples: {X_test_scaled.shape[0]:,}

MODEL PERFORMANCE:
Best Model: Random Forest
- AUC-ROC Score: {results['Random Forest']['auc']:.4f}
- F1-Score: {results['Random Forest']['f1']:.4f}

KEY FINDINGS:
1. Usage patterns are critical indicators of churn
2. Telecom partner influences customer retention
3. Age groups show varying churn rates
4. Long-term customers (high days_active) have lower churn

RECOMMENDATION:
Implement the predictive model in production and establish:
- Real-time churn risk monitoring
- Automated alerts for high-risk customers
- Targeted retention campaigns
- Quarterly model retraining with new data

{'='*70}
"""

print(summary_text)